# Gráfico da Taxa por Regional de Saúde

Para cada variável, agrega por Regional de Saúde somando todo o
período disponível, e gera um gráfico com barras (valor absoluto,
eixo esquerdo) e linha (taxa por 100 mil habitantes, eixo direito),
ordenado da Regional com maior taxa para a de menor taxa.



## Imports

In [1]:
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

from utilitarios_populacional import (
    COLUNA_CODIGO_IBGE,
    COLUNA_REGIONAL,
    COLUNA_ANO,
    COLUNA_TIPO,
    COLUNA_QUANTIDADE,
    COLUNA_POPULACAO,
    PREFIXO_COLUNA_TOTAL_ANO,
    MULTIPLICADOR_TAXA,
    registrar_log,
    normalizar_codigo_ibge,
    carregar_taxa,
    montar_grafico_barra_linha,
    montar_grafico_empilhado,
    salvar_figura,
    melt_variavel_categorica,
    ordenar_categorias,
    gerar_paleta_cores,
)

## Configuração Geral

In [ ]:
PASTA_TAXAS = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional")
CAMINHO_POPULACAO = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/populacao_interpolacao_extrapolacao_geometrica.csv")
CAMINHO_REGIONAIS = Path("/home/anna/tcc/Seleção e Transformação/Municípios/tabela_municipios_regionais.csv")

PASTA_SAIDA = Path("/home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras")

ORDENAR_POR_TAXA_DESCENDENTE = True

LARGURA_GRAFICO = 1300
ALTURA_GRAFICO = 650

LISTA_VARIAVEIS = [
    {
        "NOME": "Casos",
        "CAMINHO_ENTRADA": PASTA_TAXAS / "casos_taxa_por_100_mil.csv",
        "ANO_INICIAL": 1985,
        "ANO_FINAL": 2023,
        "TITULO_GRAFICO": "Incidência de Câncer de Mama por Regional de Saúde em Minas Gerais",
        "NOME_ABSOLUTO": "Casos Absolutos",
        "COR_BARRAS": "#7b2d8e",
        "COR_LINHA": "#2a9d8f",
        "NOME_ARQUIVO_SAIDA": "taxa_casos_por_regional",
    },
    {
        "NOME": "Mortalidade",
        "CAMINHO_ENTRADA": PASTA_TAXAS / "mortalidade_taxa_por_100_mil.csv",
        "ANO_INICIAL": 1979,
        "ANO_FINAL": 2024,
        "TITULO_GRAFICO": "Mortalidade por Câncer de Mama por Regional de Saúde em Minas Gerais",
        "NOME_ABSOLUTO": "Óbitos Absolutos",
        "COR_BARRAS": "#c0392b",
        "COR_LINHA": "#2c3e50",
        "NOME_ARQUIVO_SAIDA": "taxa_mortalidade_por_regional",
    },
    {
        "NOME": "Mamografia",
        "CAMINHO_ENTRADA": PASTA_TAXAS / "mamografia_taxa_por_100_mil.csv",
        "ANO_INICIAL": 2013,
        "ANO_FINAL": 2025,
        "TITULO_GRAFICO": "Mamografias por Regional de Saúde em Minas Gerais",
        "NOME_ABSOLUTO": "Mamografias Absolutas",
        "COR_BARRAS": "#0e7c7b",
        "COR_LINHA": "#d17a22",
        "NOME_ARQUIVO_SAIDA": "taxa_mamografia_por_regional",
    },
    {
        "NOME": "Hospitais",
        "CAMINHO_ENTRADA": PASTA_TAXAS / "hospitais_taxa_por_100_mil.csv",
        "ANO_INICIAL": 2023,
        "ANO_FINAL": 2023,
        "TITULO_GRAFICO": "Hospitais Habilitados por Regional de Saúde em Minas Gerais (2023)",
        "NOME_ABSOLUTO": "Habilitações Absolutas",
        "COR_BARRAS": "#8a5a44",
        "COR_LINHA": "#1f7a4d",
        "NOME_ARQUIVO_SAIDA": "taxa_hospitais_por_regional",
    },
]

### Soma das Regionais 

$$\text{Taxa}_{\text{regional}} = \frac{\sum_{\text{ano}}\sum_{\text{município da regional}} \text{Quantidade}}{\sum_{\text{ano}}\sum_{\text{município da regional}} \text{População}} \times 100.000$$

In [ ]:
def somar_quantidade_por_regional(tabela_taxa, mapa_regional, nome_da_variavel):
    '''
    Atribui a Regional de Saúde de cada município (pelo Código IBGE) e
    soma a QUANTIDADE de todo o período, agrupando por regional.

    '''
    tabela_taxa = tabela_taxa.copy()

    linhas_sem_populacao = tabela_taxa[COLUNA_POPULACAO].isna()
    if linhas_sem_populacao.any():
        registrar_log(
            f"ATENÇÃO ({nome_da_variavel}): {linhas_sem_populacao.sum()} linha(s) sem "
            f"POPULACAO válida foram excluídas da soma de QUANTIDADE por regional "
            f"(para não inflar a taxa agregada)."
        )
    tabela_taxa = tabela_taxa[~linhas_sem_populacao]

    tabela_taxa[COLUNA_REGIONAL] = tabela_taxa[COLUNA_CODIGO_IBGE].map(mapa_regional)

    quantidade_sem_regional = tabela_taxa[COLUNA_REGIONAL].isna().sum()
    if quantidade_sem_regional > 0:
        registrar_log(
            f"ATENÇÃO ({nome_da_variavel}): {quantidade_sem_regional} registro(s) "
            f"sem Regional de Saúde correspondente no mapeamento."
        )

    return tabela_taxa.groupby(COLUNA_REGIONAL)[COLUNA_QUANTIDADE].sum()


def somar_populacao_por_regional(tabela_populacao, mapa_regional, ano_inicial, ano_final, nome_da_variavel):
    '''
    Soma, para cada município, as colunas TOTAL_<ano> dentro do
    período informado, atribui a Regional de Saúde de cada município
    e agrega (soma) o total do período por regional.
    '''
    colunas_do_periodo = []
    for ano in range(ano_inicial, ano_final + 1):
        nome_da_coluna = f"{PREFIXO_COLUNA_TOTAL_ANO}{ano}"
        if nome_da_coluna in tabela_populacao.columns:
            colunas_do_periodo.append(nome_da_coluna)

    tabela_populacao = tabela_populacao.copy()
    tabela_populacao["TOTAL_PERIODO"] = tabela_populacao[colunas_do_periodo].sum(axis=1)
    tabela_populacao[COLUNA_REGIONAL] = tabela_populacao[COLUNA_CODIGO_IBGE].map(mapa_regional)

    quantidade_sem_regional = tabela_populacao[COLUNA_REGIONAL].isna().sum()
    if quantidade_sem_regional > 0:
        registrar_log(
            f"ATENÇÃO ({nome_da_variavel}): {quantidade_sem_regional} município(s) "
            f"sem Regional de Saúde correspondente no mapeamento de população."
        )

    return tabela_populacao.groupby(COLUNA_REGIONAL)["TOTAL_PERIODO"].sum()

## Célula 4 — Carregamento das Regionais

In [ ]:
tabela_regionais = pd.read_csv(CAMINHO_REGIONAIS)
tabela_populacao = pd.read_csv(CAMINHO_POPULACAO)

registrar_log(
    f"dtype de '{COLUNA_CODIGO_IBGE}' em tabela_regionais: {tabela_regionais[COLUNA_CODIGO_IBGE].dtype}"
)
registrar_log(
    f"dtype de '{COLUNA_CODIGO_IBGE}' em tabela_populacao: {tabela_populacao[COLUNA_CODIGO_IBGE].dtype}"
)

codigos_regionais_bruto = set(tabela_regionais[COLUNA_CODIGO_IBGE])
codigos_populacao_bruto = set(tabela_populacao[COLUNA_CODIGO_IBGE])
diferenca_bruta = codigos_populacao_bruto - codigos_regionais_bruto
if diferenca_bruta:
    registrar_log(
        f"ATENÇÃO: {len(diferenca_bruta)} código(s) IBGE de tabela_populacao não batem com "
        f"tabela_regionais ANTES da normalização de tipo: {sorted(diferenca_bruta, key=str)}"
    )
else:
    registrar_log("Nenhuma divergência de Código IBGE encontrada antes mesmo da normalização.")

tabela_regionais[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_regionais[COLUNA_CODIGO_IBGE])
tabela_populacao[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_populacao[COLUNA_CODIGO_IBGE])

codigos_regionais = set(tabela_regionais[COLUNA_CODIGO_IBGE])
codigos_populacao = set(tabela_populacao[COLUNA_CODIGO_IBGE])
diferenca_pos_normalizacao = codigos_populacao - codigos_regionais
if diferenca_pos_normalizacao:
    registrar_log(
        f"ATENÇÃO: mesmo após normalizar o tipo, {len(diferenca_pos_normalizacao)} código(s) IBGE "
        f"de tabela_populacao continuam sem correspondência em tabela_regionais — isto sim é um "
        f"mapeamento incompleto de verdade: {sorted(diferenca_pos_normalizacao, key=str)}"
    )
else:
    registrar_log(
        "Após normalizar o tipo do Código IBGE, todos os municípios de tabela_populacao têm "
        "correspondência em tabela_regionais (0 município sem regional por causa de tipo)."
    )

mapa_regional = tabela_regionais.set_index(COLUNA_CODIGO_IBGE)[COLUNA_REGIONAL]

registrar_log(f"Regionais carregadas: {tabela_regionais.shape}")
registrar_log(f"População carregada: {tabela_populacao.shape}")

[LOG] dtype de 'CÓDIGO IBGE' em tabela_regionais: int64
[LOG] dtype de 'CÓDIGO IBGE' em tabela_populacao: int64
[LOG] Nenhuma divergência de Código IBGE encontrada antes mesmo da normalização.
[LOG] Após normalizar o tipo do Código IBGE, todos os municípios de tabela_populacao têm correspondência em tabela_regionais (0 município sem regional por causa de tipo).
[LOG] Regionais carregadas: (853, 5)
[LOG] População carregada: (853, 58)


##  Processamento e Geração dos Gráficos


In [5]:
dicionario_figuras_por_variavel = {}

for configuracao_da_variavel in LISTA_VARIAVEIS:

    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO: {nome_da_variavel.upper()} =====")

    tabela_taxa = carregar_taxa(
        configuracao_da_variavel["CAMINHO_ENTRADA"],
        configuracao_da_variavel["ANO_INICIAL"],
        configuracao_da_variavel["ANO_FINAL"],
        normalizar_ibge=True,
    )

    quantidade_por_regional = somar_quantidade_por_regional(tabela_taxa, mapa_regional, nome_da_variavel)

    populacao_por_regional = somar_populacao_por_regional(
        tabela_populacao,
        mapa_regional,
        configuracao_da_variavel["ANO_INICIAL"],
        configuracao_da_variavel["ANO_FINAL"],
        nome_da_variavel,
    )

    todas_as_regionais = sorted(set(quantidade_por_regional.index) | set(populacao_por_regional.index))
    quantidade_por_regional = quantidade_por_regional.reindex(todas_as_regionais, fill_value=0)
    populacao_por_regional = populacao_por_regional.reindex(todas_as_regionais)

    taxa_por_regional = ((quantidade_por_regional / populacao_por_regional) * MULTIPLICADOR_TAXA).round(1)

    if ORDENAR_POR_TAXA_DESCENDENTE:
        ordem_das_regionais = taxa_por_regional.sort_values(ascending=False).index
        quantidade_por_regional = quantidade_por_regional.reindex(ordem_das_regionais)
        taxa_por_regional = taxa_por_regional.reindex(ordem_das_regionais)

    nomes_das_regionais = [nome.title() for nome in taxa_por_regional.index]

    figura_da_variavel = montar_grafico_barra_linha(
        eixo_x=nomes_das_regionais,
        valores_absolutos=quantidade_por_regional.values,
        valores_taxa=taxa_por_regional.values,
        titulo=configuracao_da_variavel["TITULO_GRAFICO"],
        nome_absoluto=configuracao_da_variavel["NOME_ABSOLUTO"],
        cor_barras=configuracao_da_variavel["COR_BARRAS"],
        cor_linha=configuracao_da_variavel["COR_LINHA"],
        titulo_eixo_x="Regional de Saúde",
        largura=LARGURA_GRAFICO,
        altura=ALTURA_GRAFICO,
        tickangle_eixo_x=-45,
        margem_inferior=140,
        legenda_y=-0.3,
    )

    salvar_figura(figura_da_variavel, configuracao_da_variavel["NOME_ARQUIVO_SAIDA"], PASTA_SAIDA)

    dicionario_figuras_por_variavel[nome_da_variavel] = figura_da_variavel

    registrar_log(f"===== FIM: {nome_da_variavel.upper()} =====\n")

[LOG] ===== INÍCIO: CASOS =====
[LOG] ATENÇÃO (Casos): 780 linha(s) sem POPULACAO válida foram excluídas da soma de QUANTIDADE por regional (para não inflar a taxa agregada).


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_casos_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_casos_por_regional.html
[LOG] ===== FIM: CASOS =====

[LOG] ===== INÍCIO: MORTALIDADE =====
[LOG] ATENÇÃO (Mortalidade): 1387 linha(s) sem POPULACAO válida foram excluídas da soma de QUANTIDADE por regional (para não inflar a taxa agregada).


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mortalidade_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mortalidade_por_regional.html
[LOG] ===== FIM: MORTALIDADE =====

[LOG] ===== INÍCIO: MAMOGRAFIA =====
[LOG] ATENÇÃO (Mamografia): 78 linha(s) sem POPULACAO válida foram excluídas da soma de QUANTIDADE por regional (para não inflar a taxa agregada).


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mamografia_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_mamografia_por_regional.html
[LOG] ===== FIM: MAMOGRAFIA =====

[LOG] ===== INÍCIO: HOSPITAIS =====


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_hospitais_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_hospitais_por_regional.html
[LOG] ===== FIM: HOSPITAIS =====



## Exibição: Casos

In [6]:
dicionario_figuras_por_variavel["Casos"].show()

## Célula 7 — Exibição: Mortalidade

In [7]:
dicionario_figuras_por_variavel["Mortalidade"].show()

## Célula 8 — Exibição: Mamografia

In [8]:
dicionario_figuras_por_variavel["Mamografia"].show()

## Célula 9 — Exibição: Hospitais (total, todos os tipos somados)

In [9]:
dicionario_figuras_por_variavel["Hospitais"].show()

---
## Hospitais por Tipo

In [10]:
CAMINHO_HOSPITAIS_POR_TIPO = PASTA_TAXAS / "hospitais_por_tipo_taxa_por_100_mil.csv"

tabela_hospitais_por_tipo = carregar_taxa(CAMINHO_HOSPITAIS_POR_TIPO, normalizar_ibge=True)
tabela_hospitais_por_tipo[COLUNA_REGIONAL] = tabela_hospitais_por_tipo[COLUNA_CODIGO_IBGE].map(mapa_regional)

quantidade_por_regional_e_tipo = (
    tabela_hospitais_por_tipo
    .groupby([COLUNA_REGIONAL, COLUNA_TIPO])[COLUNA_QUANTIDADE]
    .sum()
    .unstack(COLUNA_TIPO, fill_value=0)
)

populacao_por_regional_hospitais = somar_populacao_por_regional(
    tabela_populacao, mapa_regional, 2023, 2023, "Hospitais por Tipo"
)
populacao_por_regional_hospitais = populacao_por_regional_hospitais.reindex(quantidade_por_regional_e_tipo.index)

taxa_total_por_regional = (
    (quantidade_por_regional_e_tipo.sum(axis=1) / populacao_por_regional_hospitais) * MULTIPLICADOR_TAXA
).round(1)

ordem_das_regionais_hospitais = taxa_total_por_regional.sort_values(ascending=False).index
quantidade_por_regional_e_tipo = quantidade_por_regional_e_tipo.reindex(ordem_das_regionais_hospitais)
taxa_total_por_regional = taxa_total_por_regional.reindex(ordem_das_regionais_hospitais)

nomes_das_regionais_hospitais = [nome.title() for nome in quantidade_por_regional_e_tipo.index]

CORES_POR_TIPO = ["#8a5a44", "#c9a66b", "#5c8d89"]

figura_hospitais_por_tipo = go.Figure()

for indice_tipo, tipo in enumerate(quantidade_por_regional_e_tipo.columns):
    figura_hospitais_por_tipo.add_trace(go.Bar(
        x=nomes_das_regionais_hospitais,
        y=quantidade_por_regional_e_tipo[tipo].values,
        name=tipo.title(),
        marker_color=CORES_POR_TIPO[indice_tipo % len(CORES_POR_TIPO)],
        yaxis="y1",
    ))

figura_hospitais_por_tipo.add_trace(go.Scatter(
    x=nomes_das_regionais_hospitais,
    y=taxa_total_por_regional.values,
    name="Taxa Total Ajustada — /100k hab. (eixo dir.)",
    mode="lines+markers+text",
    marker=dict(color="#1f7a4d", size=7),
    line=dict(color="#1f7a4d", width=3),
    text=[f"{v:.1f}" for v in taxa_total_por_regional.values],
    textposition="top center",
    textfont=dict(size=13, color="#1f7a4d"),
    yaxis="y2",
))

figura_hospitais_por_tipo.update_layout(
    title="Hospitais Habilitados por Tipo, por Regional de Saúde em Minas Gerais (2023)",
    template="plotly_white",
    barmode="group",
    width=LARGURA_GRAFICO,
    height=ALTURA_GRAFICO,
    margin=dict(l=60, r=60, t=100, b=140),
    xaxis=dict(title="Regional de Saúde", tickangle=-45),
    yaxis=dict(title="Habilitações Absolutas", side="left"),
    yaxis2=dict(title="Taxa Total / 100k hab.", overlaying="y", side="right", showgrid=False),
    legend=dict(orientation="h", yanchor="top", y=-0.3, xanchor="center", x=0.5),
)

salvar_figura(figura_hospitais_por_tipo, "taxa_hospitais_por_tipo_por_regional", PASTA_SAIDA)

figura_hospitais_por_tipo.show()

[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_hospitais_por_tipo_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/taxa_hospitais_por_tipo_por_regional.html


##  Configuração dos Gráficos Empilhados

In [ ]:

CATEGORIAS_EXCLUIDAS = ()   # ex.: ("SEM INFORMAÇÃO",) para excluir essa categoria dos gráficos

ORDENAR_REGIONAIS_POR_TOTAL_DESCENDENTE = True

LARGURA_GRAFICO_EMPILHADO = LARGURA_GRAFICO
ALTURA_GRAFICO_EMPILHADO = ALTURA_GRAFICO

LISTA_VARIAVEIS_CATEGORICAS = [
    {
        "NOME": "Faixa Etária",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _faixa_etaria.csv"),
        "TITULO_GRAFICO": "Composição dos Casos de Câncer de Mama por Faixa Etária, por Regional de Saúde",
        "ORDEM_CATEGORIAS": ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70 OU MAIS"],
        "NOME_ARQUIVO_SAIDA": "empilhado_faixa_etaria_por_regional",
    },
    {
        "NOME": "Raça/Cor",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _raca_cor.csv"),
        "TITULO_GRAFICO": "Composição dos Casos de Câncer de Mama por Raça/Cor, por Regional de Saúde",
        "ORDEM_CATEGORIAS": None,   # None = ordem alfabética, com SEM INFORMAÇÃO por último
        "NOME_ARQUIVO_SAIDA": "empilhado_raca_cor_por_regional",
    },
    {
        "NOME": "Escolaridade",
        "CAMINHO_ENTRADA": Path("/home/anna/tcc/Seleção e Transformação/Casos/incidencia _escolaridade.csv"),
        "TITULO_GRAFICO": "Composição dos Casos de Câncer de Mama por Escolaridade, por Regional de Saúde",
        "ORDEM_CATEGORIAS": [
            "NENHUMA", "FUNDAMENTAL INCOMPLETO", "FUNDAMENTAL COMPLETO", "NÍVEL MÉDIO",
            "NÍVEL SUPERIOR INCOMPLETO", "NÍVEL SUPERIOR COMPLETO", "SEM INFORMAÇÃO",
        ],
        "NOME_ARQUIVO_SAIDA": "empilhado_escolaridade_por_regional",
    },
]

## Processamento e Geração dos Gráficos Empilhados

In [12]:
dicionario_figuras_categoricas = {}

for configuracao_da_variavel in LISTA_VARIAVEIS_CATEGORICAS:

    nome_da_variavel = configuracao_da_variavel["NOME"]
    registrar_log(f"===== INÍCIO (empilhado): {nome_da_variavel.upper()} =====")

    tabela_bruta = pd.read_csv(configuracao_da_variavel["CAMINHO_ENTRADA"])
    tabela_bruta[COLUNA_CODIGO_IBGE] = normalizar_codigo_ibge(tabela_bruta[COLUNA_CODIGO_IBGE])
    tabela_longa = melt_variavel_categorica(tabela_bruta, COLUNA_CODIGO_IBGE)

    if CATEGORIAS_EXCLUIDAS:
        tabela_longa = tabela_longa[~tabela_longa["CATEGORIA"].isin(CATEGORIAS_EXCLUIDAS)]

    tabela_longa[COLUNA_REGIONAL] = tabela_longa[COLUNA_CODIGO_IBGE].map(mapa_regional)

    quantidade_sem_regional = tabela_longa[COLUNA_REGIONAL].isna().sum()
    if quantidade_sem_regional > 0:
        registrar_log(
            f"ATENÇÃO ({nome_da_variavel}): {quantidade_sem_regional} registro(s) sem "
            f"Regional de Saúde correspondente."
        )

    tabela_agregada = tabela_longa.groupby([COLUNA_REGIONAL, "CATEGORIA"], as_index=False)["VALOR"].sum()
    tabela_pivotada = tabela_agregada.pivot(index=COLUNA_REGIONAL, columns="CATEGORIA", values="VALOR").fillna(0)

    ordem_categorias = ordenar_categorias(
        tabela_pivotada.columns.tolist(), configuracao_da_variavel["ORDEM_CATEGORIAS"]
    )
    paleta_cores = gerar_paleta_cores(ordem_categorias)

    if ORDENAR_REGIONAIS_POR_TOTAL_DESCENDENTE:
        ordem_regionais = tabela_pivotada[ordem_categorias].sum(axis=1).sort_values(ascending=False).index
        tabela_pivotada = tabela_pivotada.reindex(ordem_regionais)

    nomes_das_regionais = [nome.title() for nome in tabela_pivotada.index]

    figura_da_variavel = montar_grafico_empilhado(
        eixo_x=nomes_das_regionais,
        tabela_pivotada=tabela_pivotada,
        titulo=configuracao_da_variavel["TITULO_GRAFICO"],
        paleta_cores=paleta_cores,
        titulo_eixo_x="Regional de Saúde",
        ordem_categorias=ordem_categorias,
        largura=LARGURA_GRAFICO_EMPILHADO,
        altura=ALTURA_GRAFICO_EMPILHADO,
        tickangle_eixo_x=-45,
        margem_inferior=140,
        legenda_y=-0.3,
    )

    salvar_figura(figura_da_variavel, configuracao_da_variavel["NOME_ARQUIVO_SAIDA"], PASTA_SAIDA)

    dicionario_figuras_categoricas[nome_da_variavel] = figura_da_variavel

    registrar_log(f"===== FIM (empilhado): {nome_da_variavel.upper()} =====\n")

[LOG] ===== INÍCIO (empilhado): FAIXA ETÁRIA =====


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_faixa_etaria_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_faixa_etaria_por_regional.html
[LOG] ===== FIM (empilhado): FAIXA ETÁRIA =====

[LOG] ===== INÍCIO (empilhado): RAÇA/COR =====


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_raca_cor_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_raca_cor_por_regional.html
[LOG] ===== FIM (empilhado): RAÇA/COR =====

[LOG] ===== INÍCIO (empilhado): ESCOLARIDADE =====


[LOG] Figura PNG salva em : /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_escolaridade_por_regional.png
[LOG] Figura HTML salva em: /home/anna/tcc/Mineração de Dados - Avaliação e Apresentação/Análise Populacional/Figuras/empilhado_escolaridade_por_regional.html
[LOG] ===== FIM (empilhado): ESCOLARIDADE =====



## Célula 14 — Exibição: Faixa Etária

In [13]:
dicionario_figuras_categoricas["Faixa Etária"].show()

## Célula 15 — Exibição: Raça/Cor

In [14]:
dicionario_figuras_categoricas["Raça/Cor"].show()

## Célula 16 — Exibição: Escolaridade

In [15]:
dicionario_figuras_categoricas["Escolaridade"].show()